In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import subprocess
import pandas as pd
from strip_ansi import strip_ansi
from IPython.display import display
from lxml import etree
from conlanger.tools.SoundChangeRule import SoundChangeRule, DebugRules

RULES_DIR = "./data/rules"
DEBUG_RULES_DIR = "./data/rules/{format}/tmp"
OVERWRITE_RULES = False
CREATE_DEBUG_RULES = True
# RULES_XML = "./data/index_diachronica.xml"
RULES_XML = "./data/output.xml"

In [ ]:
def run_asca_debug(asca_word_file, rule_file):
    file_name = f"{DEBUG_RULES_DIR.format(format="asca")}/{rule_file}"
    print(file_name)
    cmd = f"~/.cargo/bin/asca run {asca_word_file} --rules {file_name}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
        output.check_returncode()

    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = strip_ansi(exc.stderr.strip()).replace('\n', ' ')
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace('\n', ' ')

    return result

In [ ]:
asca_debug_results = []
asca_debug_word_file = "./data/words/asca/weirdness_0.5.wsca"

if CREATE_DEBUG_RULES:
    tree = etree.parse(RULES_XML)
    root = tree.getroot()
    

    debug_rules = []

    for child in root:
        index = child.attrib["index"]
        if len(child.findall("rule")) > 0:
            for r in DebugRules(child, format="asca").rules:
                debug_rules.append(((r.title + '.rsca').replace(' ', '_'), str(r)))

    for name, rule in debug_rules[:100]:
        file_name = f"{DEBUG_RULES_DIR.format(format="asca")}/{name}"
        with open(file_name, "w") as f:
            f.write(rule)

        result = run_asca_debug(asca_debug_word_file, name)
        asca_debug_results.append(result)

        os.remove(file_name)

    asca_debug_results_df = pd.DataFrame(asca_debug_results)

    print(asca_debug_results_df["returncode"].value_counts())

    errors = asca_debug_results_df[asca_debug_results_df["returncode"] != 0][["rule", "error"]]

    for r in errors.itertuples(index=False):
        print(r.rule, r.error)
                

In [ ]:
if OVERWRITE_RULES:
    tree = etree.parse(RULES_XML)
    root = tree.getroot()

    results = []
    rules = []

    for child in root:
        index = child.attrib["index"]
        results.append({
            "index": index,
            "name": child.attrib["name"],
            "rule_count": len(child.findall("rule")),
        })

        rules.append((index, str(SoundChangeRule(child, format="asca"))))


    sections_df = pd.DataFrame(results)

    sections_df.to_csv("./data/index_diachronica_sections.csv", index=False)
    
    for index, rule in rules:
        with open(f"./data/rules/asca/{index}.rsca", "w") as f:
            f.write(rule)

    sections_df.head(3)

In [ ]:
sections_df = pd.read_csv("./data/index_diachronica_sections.csv", dtype={"index": str, "name": str, "rule_count": int})

rules_df = sections_df[sections_df["rule_count"] > 0].copy()
rules_df['asca_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.rsca")
rules_df['brassica_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.bsc")

display(rules_df.head(3))

asca_rule_files = rules_df['asca_rule_file'].tolist()
brassica_rule_files = rules_df['brassica_rule_file'].tolist()


In [ ]:
# run asca-rust to validate rules

asca_results = []
asca_word_file = "./data/words/asca/weirdness_0.5.wsca"
asca_alias_file = "./data/asca_aliases.alias"

def run_asca(asca_word_file, rule_file):
    cmd = f"~/.cargo/bin/asca run {asca_word_file} --rules {RULES_DIR}/asca/{rule_file}"# --alias {asca_alias_file}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
        output.check_returncode()

    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = strip_ansi(exc.stderr.strip()).replace('\n', ' ')
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace('\n', ' ')

    return result

for rule_file in asca_rule_files[:100]:
    result = run_asca(asca_word_file, rule_file)
    asca_results.append(result)

asca_results_df = pd.DataFrame(asca_results)

asca_results_df.to_csv("./data/asca_results.csv", index=False)

print(asca_results_df["returncode"].value_counts())

errors = asca_results_df[asca_results_df["returncode"] != 0][["rule", "error"]]

for r in errors.itertuples(index=False):
    print(r.rule, r.error)

In [ ]:
# run Brassica to validate rules

# brassica_results = []
# brassica_word_file = "./data/words/brassica/weirdness_0.5.lex"

# def run_brassica(brassica_word_file, rule_file):
#     cmd = f"brassica {RULES_DIR}/brassica/{rule_file} -i {brassica_word_file}"

#     result = {"rule": rule_file, "returncode": 0, "error": ""}

#     try:
#         print(cmd)
#         output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
#         print(output.stdout)
#         output.check_returncode()

#     except subprocess.CalledProcessError as exc:
#         result["returncode"] = exc.returncode 
#         result["error"] = exc.output.replace("\n", "\\n")
#     except subprocess.TimeoutExpired as exc:
#         result["returncode"] = 124
#         result["error"] = exc.output.decode("utf-8").replace("\n", "\\n")

#     return result

# for rule_file in brassica_rule_files[:10]:
#     result = run_brassica(brassica_word_file, rule_file)
#     brassica_results.append(result)

# brassica_results_df = pd.DataFrame(brassica_results)

# brassica_results_df.to_csv("./data/brassica_results.csv", index=False)

# brassica_results_df[brassica_results_df["returncode"] != 0].head()